# Projet IA — Planification personnalisée de séjours

**Objectif du notebook :** préparer le dataset de voyages, traiter les incohérences critiques, construire un premier pipeline de modélisation et évaluer un modèle baseline pour prédire la satisfaction client.

## 0. Contexte et périmètre

Une agence de voyages haut de gamme souhaite concevoir une solution IA capable de personnaliser la planification de séjours.

Le modèle principal étudié dans ce notebook vise à prédire `satisfaction_client` à partir des informations disponibles avant ou pendant la planification du voyage.

Variables post-séjour exclues du modèle principal :

- `imprevus` ;
- `reorganisation_necessaire` ;
- `respect_budget` ;
- `retour_client`.

Ces variables restent utiles pour l'analyse métier, mais elles ne doivent pas être utilisées comme entrées du modèle principal afin d'éviter une fuite de données.

## 1. Datasheet synthétique

| Élément | Description |
| --- | --- |
| Fichier | `data/Examen_travel_planning_dataset.csv` |
| Format | CSV |
| Volume attendu | Environ 1500 séjours |
| Domaine | Planification de voyages haut de gamme |
| Nature des données | Données synthétiques et anonymisées |
| Cible IA principale | `satisfaction_client` |
| Problème ML | Classification multi-classes de satisfaction |

Le fichier contient des informations sur le profil client, le budget, la destination, la saison, la durée, l'hébergement, le vol, la météo prévue, l'activité principale et les résultats observés après séjour.

## 2. Configuration

Cette section importe les bibliothèques nécessaires et limite le nombre de threads pour réduire la consommation mémoire sur une machine disposant de peu de RAM.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)
sns.set_theme(style="whitegrid")

## 3. Chargement du dataset brut

Le dataset brut est conservé dans `df_raw`. Les traitements de préparation seront appliqués sur une copie afin de garder une référence intacte aux données originales.

In [ ]:
filename = "Examen_travel_planning_dataset.csv"
candidate_paths = [
    Path("data") / filename,
    Path("data/raw") / filename,
    Path("../data") / filename,
    Path("../data/raw") / filename,
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(f"Fichier introuvable : {filename}")

df_raw = pd.read_csv(data_path)

print(f"Fichier chargé : {data_path}")
print(f"Dimensions : {df_raw.shape[0]} lignes x {df_raw.shape[1]} colonnes")
display(df_raw.head())

## 4. Exploration rapide du brut

Cette exploration vérifie la structure du fichier avant toute modification : dimensions, types, valeurs manquantes et doublons.

In [ ]:
resume_brut = pd.DataFrame({
    "type": df_raw.dtypes.astype(str),
    "valeurs_manquantes": df_raw.isna().sum(),
    "pourcentage_manquant": (df_raw.isna().sum() / len(df_raw) * 100).round(2),
    "valeurs_uniques": df_raw.nunique(dropna=True),
})

display(resume_brut)

print("Doublons exacts :", int(df_raw.duplicated().sum()))
print("Doublons trip_id :", int(df_raw["trip_id"].duplicated().sum()))

In [ ]:
missing_summary = (
    df_raw.isna().sum()
    .to_frame("valeurs_manquantes")
)
missing_summary["pourcentage"] = (missing_summary["valeurs_manquantes"] / len(df_raw) * 100).round(2)
missing_summary = missing_summary[missing_summary["valeurs_manquantes"] > 0].sort_values("valeurs_manquantes", ascending=False)

display(missing_summary)

if not missing_summary.empty:
    plt.figure(figsize=(9, 5))
    plt.barh(missing_summary.index[::-1], missing_summary["pourcentage"][::-1], color="coral")
    plt.xlabel("Pourcentage de valeurs manquantes (%)")
    plt.title("Valeurs manquantes par colonne")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Analyse métier de cohérence du dataset brut

Cette section reprend l'analyse métier détaillée du notebook `analyse_métier.ipynb`.

Elle est placée avant le nettoyage pour distinguer :

- les incohérences critiques à traiter avant la modélisation ;
- les points de vigilance à documenter ;
- les variables à exclure du modèle principal pour éviter la fuite de données.


In [ ]:
# Fonctions utilitaires utilisées dans tout le notebook

def normaliser_texte(serie):
    """Standardise les textes pour faciliter les comparaisons métier."""
    return serie.fillna("").astype(str).str.strip().str.lower()


def afficher_cas(df_cas, colonnes=None, n=10):
    """Affiche le nombre de cas détectés puis un échantillon lisible."""
    print(f"Nombre de cas détectés : {len(df_cas)}")
    if len(df_cas) == 0:
        print("Aucun cas à afficher.")
        return

    if colonnes is not None:
        display(df_cas[colonnes].head(n))
    else:
        display(df_cas.head(n))


colonnes_principales = [
    "trip_id", "client_type", "budget_total", "destination", "saison",
    "duree_jours", "type_hebergement", "prix_vol", "meteo_prevue",
    "activite_principale", "satisfaction_client", "imprevus",
    "reorganisation_necessaire", "respect_budget"
]

### 5.1. Unicité des séjours

### Observation métier

Chaque ligne représente un séjour passé. La colonne `trip_id` doit donc identifier un séjour unique.

Un doublon sur `trip_id` pourrait fausser l'analyse car un même séjour serait compté plusieurs fois.

In [ ]:
doublons_trip_id = df_raw[df_raw["trip_id"].duplicated(keep=False)].copy()
doublons_lignes = df_raw[df_raw.duplicated(keep=False)].copy()

print("Doublons sur trip_id :")
afficher_cas(doublons_trip_id, colonnes_principales)

print("\nDoublons exacts sur toutes les colonnes :")
afficher_cas(doublons_lignes, colonnes_principales)

### 5.2. Cohérence de la satisfaction client

### Observation métier

Le cahier des charges indique que `satisfaction_client` est un score de 1 à 5.

Les valeurs inférieures à 1, supérieures à 5 ou manquantes sont donc incohérentes avec la définition officielle de la variable.

Ces cas doivent être corrigés, exclus ou documentés avant d'utiliser `satisfaction_client` comme cible IA.

In [ ]:
satisfaction_invalide = df_raw[
    df_raw["satisfaction_client"].isna()
    | ~df_raw["satisfaction_client"].between(1, 5)
].copy()

afficher_cas(
    satisfaction_invalide,
    ["trip_id", "client_type", "destination", "satisfaction_client", "imprevus", "retour_client"]
)

### 5.3. Prix du vol supérieur au budget total

### Observation métier

Le `budget_total` représente le budget global du séjour.

Si `prix_vol > budget_total`, le coût du vol dépasse déjà le budget disponible. C'est une incohérence forte ou un séjour impossible à financer sans dépassement.

Le cas est encore plus problématique si `respect_budget = 1`, car cela indique que le budget aurait été respecté malgré un vol supérieur au budget total.

In [ ]:
vol_superieur_budget = df_raw[
    df_raw["prix_vol"].notna()
    & df_raw["budget_total"].notna()
    & (df_raw["prix_vol"] > df_raw["budget_total"])
].copy()

vol_superieur_budget["ecart_vol_budget"] = (
    vol_superieur_budget["prix_vol"] - vol_superieur_budget["budget_total"]
).round(2)

print("Tous les cas où le prix du vol dépasse le budget total :")
afficher_cas(
    vol_superieur_budget.sort_values("ecart_vol_budget", ascending=False),
    ["trip_id", "client_type", "destination", "budget_total", "prix_vol", "ecart_vol_budget", "respect_budget"]
)

print("\nCas contradictoires : prix_vol > budget_total ET respect_budget = 1")
vol_superieur_budget_respecte = vol_superieur_budget[vol_superieur_budget["respect_budget"] == 1]
afficher_cas(
    vol_superieur_budget_respecte,
    ["trip_id", "client_type", "destination", "budget_total", "prix_vol", "ecart_vol_budget", "respect_budget"]
)

### 5.4. Cohérence entre client business et activité principale

### Observation métier

Un client `business` peut avoir une activité principale `business`, mais ce n'est pas obligatoire : il peut aussi prolonger son séjour avec de la culture, de la gastronomie ou une activité de loisir.

Ce contrôle n'est donc pas une erreur automatique. C'est un point de vigilance pour vérifier si le dataset décrit bien le comportement attendu des voyageurs professionnels.

In [ ]:
client_type_norm = normaliser_texte(df_raw["client_type"])
activite_norm = normaliser_texte(df_raw["activite_principale"])

business_activite_non_business = df_raw[
    (client_type_norm == "business")
    & (activite_norm != "business")
    & (activite_norm != "")
].copy()

non_business_activite_business = df_raw[
    (client_type_norm != "business")
    & (client_type_norm != "")
    & (activite_norm == "business")
].copy()

print("Clients business avec une activité principale non-business :")
afficher_cas(
    business_activite_non_business,
    ["trip_id", "client_type", "destination", "saison", "activite_principale", "budget_total", "satisfaction_client"]
)

print("\nClients non-business avec une activité principale business :")
afficher_cas(
    non_business_activite_business,
    ["trip_id", "client_type", "destination", "saison", "activite_principale", "budget_total", "satisfaction_client"]
)

### 5.5. Activités extérieures et météo risquée

### Observation métier

L'activité `randonnée` est sensible à la météo.

Si la météo prévue est `pluie` ou `variable`, le séjour peut nécessiter une alternative ou un plan de réorganisation.

Ce n'est pas forcément une erreur, mais c'est une information importante pour l'anticipation des imprévus.

In [ ]:
meteo_norm = normaliser_texte(df_raw["meteo_prevue"])
activites_exterieures = ["randonnée", "randonnee"]
meteos_risquees = ["pluie", "variable"]

activites_meteo_risque = df_raw[
    activite_norm.isin(activites_exterieures)
    & meteo_norm.isin(meteos_risquees)
].copy()

afficher_cas(
    activites_meteo_risque,
    ["trip_id", "destination", "saison", "meteo_prevue", "activite_principale", "imprevus", "reorganisation_necessaire", "satisfaction_client"]
)

### 5.6. Cohérence entre imprévus et réorganisation

### Observation métier

La variable `reorganisation_necessaire` doit normalement être liée aux imprévus.

Deux situations sont à contrôler :

1. `imprevus = aucun` mais `reorganisation_necessaire = 1` : cas suspect, car il y a une réorganisation sans imprévu déclaré ;
2. `imprevus != aucun` mais `reorganisation_necessaire = 0` : cas possible si l'imprévu est mineur, mais à vérifier.

In [ ]:
imprevus_norm = normaliser_texte(df_raw["imprevus"])

aucun_imprevu_mais_reorganisation = df_raw[
    (imprevus_norm == "aucun")
    & (df_raw["reorganisation_necessaire"] == 1)
].copy()

imprevu_sans_reorganisation = df_raw[
    (imprevus_norm != "")
    & (imprevus_norm != "aucun")
    & (df_raw["reorganisation_necessaire"] == 0)
].copy()

print("Aucun imprévu déclaré mais réorganisation nécessaire :")
afficher_cas(
    aucun_imprevu_mais_reorganisation,
    ["trip_id", "destination", "meteo_prevue", "activite_principale", "imprevus", "reorganisation_necessaire", "satisfaction_client"]
)

print("\nImprévu déclaré mais aucune réorganisation :")
afficher_cas(
    imprevu_sans_reorganisation,
    ["trip_id", "destination", "meteo_prevue", "activite_principale", "imprevus", "reorganisation_necessaire", "satisfaction_client"]
)

### 5.7. Imprévus et satisfaction faible

### Observation métier

Un imprévu peut réduire la satisfaction client.

Les cas où `imprevus != aucun` et `satisfaction_client <= 2` sont cohérents métier, mais importants pour comprendre les facteurs d'insatisfaction.

Ils peuvent alimenter l'analyse explicative, mais il faut éviter d'utiliser les imprévus comme variable d'entrée si l'objectif est de prédire la satisfaction avant le départ.

In [ ]:
imprevus_satisfaction_faible = df_raw[
    (imprevus_norm != "")
    & (imprevus_norm != "aucun")
    & df_raw["satisfaction_client"].notna()
    & (df_raw["satisfaction_client"] <= 2)
].copy()

afficher_cas(
    imprevus_satisfaction_faible,
    ["trip_id", "client_type", "destination", "imprevus", "reorganisation_necessaire", "respect_budget", "satisfaction_client", "retour_client"]
)

### 5.8. Risque de fuite de données pour la modélisation

### Observation métier

Certaines colonnes décrivent le résultat du séjour après coup :

- `imprevus` ;
- `reorganisation_necessaire` ;
- `respect_budget` ;
- `retour_client`.

Si le modèle doit prédire la satisfaction avant le départ, ces variables ne doivent pas être utilisées comme entrées, car elles ne sont pas connues au moment de la recommandation.

Elles peuvent cependant servir à une analyse après séjour ou à un modèle secondaire d'amélioration continue.

In [ ]:
colonnes_post_sejour = ["imprevus", "reorganisation_necessaire", "respect_budget", "retour_client", "satisfaction_client"]
display(df_raw[["trip_id", *colonnes_post_sejour]].head(10))

### 5.9. Synthèse des contrôles

Ce tableau résume les contrôles réalisés et le nombre de cas détectés.

Il sert de base pour décider quoi corriger, quoi conserver et quoi documenter avant la modélisation.

In [ ]:
synthese_controles = pd.DataFrame([
    {
        "controle": "Doublons trip_id",
        "nb_cas": len(doublons_trip_id),
        "niveau": "Incohérence forte si > 0",
        "action_recommandee": "Supprimer ou fusionner les doublons"
    },
    {
        "controle": "Satisfaction hors échelle ou manquante",
        "nb_cas": len(satisfaction_invalide),
        "niveau": "Incohérence forte",
        "action_recommandee": "Corriger, exclure ou imputer selon justification"
    },
    {
        "controle": "Prix du vol supérieur au budget total",
        "nb_cas": len(vol_superieur_budget),
        "niveau": "Incohérence forte",
        "action_recommandee": "Contrôler budget_total, prix_vol et respect_budget"
    },
    {
        "controle": "Prix du vol > budget total et respect_budget = 1",
        "nb_cas": len(vol_superieur_budget_respecte),
        "niveau": "Contradiction métier",
        "action_recommandee": "Corriger respect_budget ou les montants"
    },
    {
        "controle": "Client business avec activité non-business",
        "nb_cas": len(business_activite_non_business),
        "niveau": "Point de vigilance",
        "action_recommandee": "Vérifier si le séjour mixte business/loisir est attendu"
    },
    {
        "controle": "Client non-business avec activité business",
        "nb_cas": len(non_business_activite_business),
        "niveau": "Point de vigilance",
        "action_recommandee": "Vérifier la définition de l'activité principale"
    },
    {
        "controle": "Randonnée avec météo risquée",
        "nb_cas": len(activites_meteo_risque),
        "niveau": "Signal métier utile",
        "action_recommandee": "Créer une variable meteo_risque ciblée sur la randonnée"
    },
    {
        "controle": "Aucun imprévu mais réorganisation nécessaire",
        "nb_cas": len(aucun_imprevu_mais_reorganisation),
        "niveau": "Cas suspect",
        "action_recommandee": "Contrôler la cohérence entre imprevus et reorganisation_necessaire"
    },
    {
        "controle": "Imprévu déclaré sans réorganisation",
        "nb_cas": len(imprevu_sans_reorganisation),
        "niveau": "Point de vigilance",
        "action_recommandee": "Conserver si imprévu mineur, sinon corriger"
    },
    {
        "controle": "Imprévu avec satisfaction faible",
        "nb_cas": len(imprevus_satisfaction_faible),
        "niveau": "Signal explicatif",
        "action_recommandee": "Utiliser pour comprendre l'insatisfaction, attention à la fuite de données"
    },
])

display(synthese_controles)

### 5.10. Sélection des incohérences à traiter avant modélisation

Toutes les observations précédentes ne doivent pas être corrigées. Certaines sont des signaux métier utiles, d'autres sont des incohérences qui peuvent dégrader directement le modèle.

Pour le modèle principal envisagé, la cible est `satisfaction_client`. On distingue donc :

- les problèmes à traiter obligatoirement avant l'entraînement ;
- les variables à exclure pour éviter la fuite de données ;
- les points de vigilance à conserver ou transformer en variables métier.


In [ ]:
selection_traitement_modele = pd.DataFrame([
    {
        "controle": "Satisfaction hors échelle ou manquante",
        "impact_modele": "Impact critique sur la cible y",
        "decision": "Traiter obligatoirement",
        "traitement_recommande": "Supprimer les lignes sans cible ou hors échelle 1-5, sauf justification métier de recodage"
    },
    {
        "controle": "Prix du vol supérieur au budget total",
        "impact_modele": "Crée des relations budgétaires impossibles et fausse les ratios",
        "decision": "Traiter obligatoirement",
        "traitement_recommande": "Identifier les cas, corriger si possible ; sinon exclure ou créer un indicateur d'anomalie"
    },
    {
        "controle": "Prix du vol > budget total et respect_budget = 1",
        "impact_modele": "Contradiction métier forte entre budget et résultat",
        "decision": "Traiter obligatoirement si respect_budget est utilisé",
        "traitement_recommande": "Corriger respect_budget ou exclure ces lignes des modèles utilisant cette variable"
    },
    {
        "controle": "Variables post-séjour : imprevus, reorganisation_necessaire, respect_budget, retour_client",
        "impact_modele": "Risque majeur de fuite de données si la prédiction est faite avant le départ",
        "decision": "Exclure des features du modèle principal",
        "traitement_recommande": "Ne pas les mettre dans X pour prédire satisfaction_client avant voyage ; les garder pour l'analyse explicative"
    },
])

display(selection_traitement_modele)

#### Observations non traitées comme erreurs

Les contrôles suivants ne sont pas corrigés automatiquement car ils peuvent représenter des comportements métier réels :

- `client business` avec activité non-business : possible séjour mixte business/loisir ;
- `client non-business` avec activité business : possible voyage personnel avec objectif professionnel ;
- activité extérieure avec météo risquée : signal utile pour créer `meteo_risque`, pas une erreur ;
- imprévu avec satisfaction faible : signal explicatif, pas une incohérence ;
- imprévu sans réorganisation : possible si l'imprévu est mineur.


### 5.11. Conclusion métier

Le dataset est exploitable pour un prototype IA, mais plusieurs points doivent être documentés avant la modélisation :

- la cible `satisfaction_client` doit être nettoyée car certaines valeurs ne respectent pas l'échelle 1 à 5 ;
- les contradictions fortes liées au budget doivent être traitées ;
- les variables post-séjour doivent être séparées des variables disponibles avant le départ pour éviter la fuite de données ;
- les contrôles météo, activité, budget journalier et type client peuvent devenir des variables utiles de feature engineering.

Ces observations justifient une étape de préparation des données avant la création du modèle IA.

## 6. Incohérences critiques retenues pour la modélisation

L'analyse métier détaillée est documentée dans `notebooks/analyse_métier.ipynb`.

Pour le modèle principal, seules les incohérences ayant un impact direct sur l'entraînement sont traitées ici :

1. `satisfaction_client` manquante ou hors échelle `1 à 5` ;
2. `prix_vol > budget_total` ;
3. `prix_vol > budget_total` avec `respect_budget = 1`, déjà inclus dans le point précédent ;
4. variables post-séjour à exclure des variables explicatives.

In [ ]:
controle_satisfaction = df_raw[
    df_raw["satisfaction_client"].isna()
    | ~df_raw["satisfaction_client"].between(1, 5)
].copy()

controle_prix_budget = df_raw[
    df_raw["prix_vol"].notna()
    & df_raw["budget_total"].notna()
    & (df_raw["prix_vol"] > df_raw["budget_total"])
].copy()

controle_prix_budget_respect = controle_prix_budget[
    controle_prix_budget["respect_budget"] == 1
].copy()

synthese_incoherences = pd.DataFrame([
    {
        "controle": "satisfaction_client manquante ou hors échelle 1-5",
        "nb_lignes": len(controle_satisfaction),
        "traitement": "suppression des lignes concernées"
    },
    {
        "controle": "prix_vol > budget_total",
        "nb_lignes": len(controle_prix_budget),
        "traitement": "suppression des lignes concernées"
    },
    {
        "controle": "prix_vol > budget_total et respect_budget = 1",
        "nb_lignes": len(controle_prix_budget_respect),
        "traitement": "déjà couvert par le contrôle prix_vol > budget_total"
    },
    {
        "controle": "variables post-séjour dans les features",
        "nb_lignes": np.nan,
        "traitement": "exclusion de X pour éviter la fuite de données"
    },
])

display(synthese_incoherences)

In [ ]:
print("Cas satisfaction_client à exclure :")
display(
    controle_satisfaction[
        ["trip_id", "client_type", "destination", "satisfaction_client", "imprevus", "retour_client"]
    ].head(10)
)

print("Cas prix_vol > budget_total à exclure :")
display(
    controle_prix_budget[
        ["trip_id", "client_type", "destination", "budget_total", "prix_vol", "respect_budget", "satisfaction_client"]
    ].sort_values(["budget_total", "prix_vol"]).head(10)
)

## 7. Préparation des données

Cette étape transforme le dataset brut nettoyé en dataset exploitable pour la modélisation.

Elle comprend :

1. le traitement des incohérences critiques ;
2. le traitement des valeurs manquantes restantes ;
3. le traitement des valeurs aberrantes numériques.

Les traitements sont appliqués avant le feature engineering afin de créer les nouvelles variables sur une base plus cohérente.


### 7.1 Traitement des incohérences critiques

Les lignes sans cible exploitable ou présentant une incohérence métier forte sont supprimées.

La cible `satisfaction_client` n'est pas imputée, car inventer une valeur cible biaiserait l'apprentissage supervisé.


In [ ]:
df_model = df_raw.copy()

nb_initial = len(df_model)

# 1. Cible valide uniquement : satisfaction_client entre 1 et 5.
mask_target_valid = (
    df_model["satisfaction_client"].notna()
    & df_model["satisfaction_client"].between(1, 5)
)
df_model = df_model[mask_target_valid].copy()
nb_after_target = len(df_model)

# 2. Cohérence budgétaire : le prix du vol ne doit pas dépasser le budget total.
mask_budget_valid = (
    df_model["prix_vol"].isna()
    | df_model["budget_total"].isna()
    | (df_model["prix_vol"] <= df_model["budget_total"])
)
df_model = df_model[mask_budget_valid].copy()
nb_after_budget = len(df_model)

rapport_nettoyage = pd.DataFrame([
    {
        "etape": "Dataset brut",
        "nb_lignes": nb_initial,
        "lignes_supprimees": 0
    },
    {
        "etape": "Cible satisfaction_client valide",
        "nb_lignes": nb_after_target,
        "lignes_supprimees": nb_initial - nb_after_target
    },
    {
        "etape": "Cohérence prix_vol <= budget_total",
        "nb_lignes": nb_after_budget,
        "lignes_supprimees": nb_after_target - nb_after_budget
    },
])

display(rapport_nettoyage)
print(f"Volume final pour modélisation : {len(df_model)} lignes")

### 7.2 Traitement des valeurs manquantes restantes

Après le traitement de la cible, les autres valeurs manquantes sont traitées selon la nature de la variable :

- variables numériques : remplacement par la médiane ;
- variables catégorielles : remplacement par la modalité `inconnu` ;
- variable textuelle `retour_client` : remplacement par une chaîne vide.

La médiane est utilisée pour les variables numériques car elle est robuste aux valeurs extrêmes. La catégorie `inconnu` permet de conserver l'information qu'une valeur était absente.


In [ ]:
missing_before_treatment = df_model.isna().sum()
missing_treatment_rows = []

numeric_missing_columns = [
    "budget_total",
    "prix_vol",
]

categorical_missing_columns = [
    "type_hebergement",
    "meteo_prevue",
    "activite_principale",
    "imprevus",
]

text_missing_columns = [
    "retour_client",
]

for column in numeric_missing_columns:
    if column in df_model.columns:
        nb_missing_before = int(df_model[column].isna().sum())
        replacement_value = df_model[column].median()
        df_model[column] = df_model[column].fillna(replacement_value)
        missing_treatment_rows.append({
            "colonne": column,
            "type_variable": "numérique",
            "valeurs_manquantes_avant": nb_missing_before,
            "traitement": "remplacement par la médiane",
            "valeur_remplacement": round(replacement_value, 2),
            "valeurs_manquantes_apres": int(df_model[column].isna().sum()),
        })

for column in categorical_missing_columns:
    if column in df_model.columns:
        nb_missing_before = int(df_model[column].isna().sum())
        df_model[column] = df_model[column].fillna("inconnu")
        missing_treatment_rows.append({
            "colonne": column,
            "type_variable": "catégorielle",
            "valeurs_manquantes_avant": nb_missing_before,
            "traitement": "remplacement par 'inconnu'",
            "valeur_remplacement": "inconnu",
            "valeurs_manquantes_apres": int(df_model[column].isna().sum()),
        })

for column in text_missing_columns:
    if column in df_model.columns:
        nb_missing_before = int(df_model[column].isna().sum())
        df_model[column] = df_model[column].fillna("")
        missing_treatment_rows.append({
            "colonne": column,
            "type_variable": "texte",
            "valeurs_manquantes_avant": nb_missing_before,
            "traitement": "remplacement par chaîne vide",
            "valeur_remplacement": "",
            "valeurs_manquantes_apres": int(df_model[column].isna().sum()),
        })

missing_treatment_report = pd.DataFrame(missing_treatment_rows)
missing_after_treatment = df_model.isna().sum()

display(missing_treatment_report)

remaining_missing = missing_after_treatment[missing_after_treatment > 0]
if remaining_missing.empty:
    print("Aucune valeur manquante restante après traitement.")
else:
    print("Valeurs manquantes restantes :")
    display(remaining_missing)

### 7.3 Traitement des valeurs aberrantes numériques

Les valeurs aberrantes sont détectées avec la méthode IQR.

Une valeur est considérée comme aberrante si elle est :

- inférieure à `Q1 - 1.5 × IQR` ;
- supérieure à `Q3 + 1.5 × IQR`.

Les colonnes traitées sont `budget_total`, `duree_jours` et `prix_vol`.

Les valeurs aberrantes sont remplacées par la médiane de leur colonne afin de limiter leur influence sur le modèle sans supprimer davantage de lignes.


In [ ]:
outlier_columns = ["budget_total", "duree_jours", "prix_vol"]
outlier_report_rows = []

for column in outlier_columns:
    q1 = df_model[column].quantile(0.25)
    q3 = df_model[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    median_value = df_model[column].median()

    outlier_mask = (
        (df_model[column] < lower_bound)
        | (df_model[column] > upper_bound)
    )
    nb_outliers = int(outlier_mask.sum())

    df_model.loc[outlier_mask, column] = median_value

    outlier_report_rows.append({
        "colonne": column,
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "IQR": round(iqr, 2),
        "borne_basse": round(lower_bound, 2),
        "borne_haute": round(upper_bound, 2),
        "mediane_remplacement": round(median_value, 2),
        "outliers_traites": nb_outliers,
        "pourcentage_lignes": round(nb_outliers / len(df_model) * 100, 2),
    })

outlier_treatment_report = pd.DataFrame(outlier_report_rows)
display(outlier_treatment_report)

# Contrôle métier après remplacement des outliers.
nb_prix_superieur_budget_apres_outliers = int((df_model["prix_vol"] > df_model["budget_total"]).sum())
print("Contrôle prix_vol > budget_total après traitement des outliers :", nb_prix_superieur_budget_apres_outliers)

## 8. Feature engineering

Les nouvelles variables sont créées uniquement à partir d'informations disponibles avant ou pendant la planification du séjour.

Elles enrichissent les dimensions budget, durée, météo, saison et profil client.

In [ ]:
df_model["budget_par_jour"] = df_model["budget_total"] / df_model["duree_jours"]
df_model["part_vol_budget"] = df_model["prix_vol"] / df_model["budget_total"]
df_model["budget_hors_vol"] = df_model["budget_total"] - df_model["prix_vol"]
df_model["sejour_long"] = (df_model["duree_jours"] >= 14).astype(int)
df_model["meteo_risque"] = df_model["meteo_prevue"].isin(["pluie", "variable"]).astype(int)
df_model["randonnee_meteo_risque"] = (
    df_model["activite_principale"].isin(["randonnée", "randonnee"])
    & df_model["meteo_prevue"].isin(["pluie", "variable"])
).astype(int)
df_model["saison_haute"] = df_model["saison"].isin(["été", "ete", "hiver"]).astype(int)
df_model["client_business"] = (df_model["client_type"] == "business").astype(int)

features_creees = [
    "budget_par_jour",
    "part_vol_budget",
    "budget_hors_vol",
    "sejour_long",
    "meteo_risque",
    "randonnee_meteo_risque",
    "saison_haute",
    "client_business",
]

for column in ["budget_par_jour", "part_vol_budget", "budget_hors_vol"]:
    df_model[column] = df_model[column].replace([np.inf, -np.inf], np.nan)

display(df_model[features_creees].head())

## 9. Enrichissement métier du dataset

Les premiers modèles montrent que les variables disponibles expliquent mal la satisfaction client.

On ajoute donc un enrichissement local par destination. L'objectif est de donner au modèle plus de contexte métier sur chaque destination, sans utiliser d'information post-séjour.

Les variables ajoutées décrivent :

- la région géographique ;
- le type de destination ;
- la distance estimée depuis l'Europe ;
- le niveau de coût de la vie ;
- le positionnement luxe ;
- le décalage horaire estimé ;
- le risque météo général par destination.

Cet enrichissement reste simple et documenté. Il sert à tester si l'ajout de contexte métier améliore les performances.

In [ ]:
enrichissement_destinations = pd.DataFrame({
    "destination": ["Paris", "Rome", "Lisbonne", "New York", "Dubaï", "Tokyo", "Bali", "Sydney"],
    "region_destination": ["Europe", "Europe", "Europe", "Amérique du Nord", "Moyen-Orient", "Asie", "Asie", "Océanie"],
    "type_destination": ["culture", "culture", "culture", "urbain_business", "luxe_shopping", "culture_urbain", "plage_luxe", "urbain_nature"],
    "distance_vol_categorie": ["court", "court", "court", "long", "moyen", "long", "long", "long"],
    "cout_vie_destination": ["élevé", "moyen", "moyen", "élevé", "élevé", "élevé", "moyen", "élevé"],
    "destination_luxe": [1, 1, 0, 1, 1, 1, 1, 1],
    "decalage_horaire_categorie": ["faible", "faible", "faible", "moyen", "moyen", "fort", "fort", "fort"],
    "risque_meteo_destination": ["moyen", "moyen", "faible", "moyen", "faible", "moyen", "élevé", "moyen"],
})

display(enrichissement_destinations)

In [ ]:
nb_lignes_avant_enrichissement = len(df_model)
nb_colonnes_avant_enrichissement = df_model.shape[1]

df_model = df_model.merge(
    enrichissement_destinations,
    on="destination",
    how="left",
    validate="many_to_one",
)

colonnes_enrichies = [
    "region_destination",
    "type_destination",
    "distance_vol_categorie",
    "cout_vie_destination",
    "destination_luxe",
    "decalage_horaire_categorie",
    "risque_meteo_destination",
]

controle_enrichissement = pd.DataFrame({
    "indicateur": [
        "lignes avant enrichissement",
        "lignes après enrichissement",
        "colonnes avant enrichissement",
        "colonnes après enrichissement",
        "valeurs manquantes sur colonnes enrichies",
    ],
    "valeur": [
        nb_lignes_avant_enrichissement,
        len(df_model),
        nb_colonnes_avant_enrichissement,
        df_model.shape[1],
        int(df_model[colonnes_enrichies].isna().sum().sum()),
    ],
})

display(controle_enrichissement)
display(df_model[["destination", *colonnes_enrichies]].drop_duplicates().sort_values("destination"))

### Limite de l'enrichissement

Cet enrichissement est créé manuellement à partir de connaissances métier générales. Il ne remplace pas des données réelles issues d'API, d'avis clients ou de systèmes de réservation.

Il permet néanmoins de tester une hypothèse importante : le modèle peut-il progresser si on ajoute du contexte métier disponible avant le départ ?

## 10. Définition de la cible et des variables explicatives

La cible est `satisfaction_client`.

Les colonnes post-séjour sont exclues de `X` pour éviter la fuite de données, car elles ne sont pas disponibles avant la réalisation du voyage.

In [ ]:
target_column = "satisfaction_client"
post_trip_columns = ["imprevus", "reorganisation_necessaire", "respect_budget", "retour_client"]
technical_columns = ["trip_id"]
excluded_columns = [target_column, *technical_columns, *post_trip_columns]

feature_columns = [column for column in df_model.columns if column not in excluded_columns]

X = df_model[feature_columns].copy()
y = df_model[target_column].astype(int).copy()
class_labels = sorted(y.unique())

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

variables_resume = pd.DataFrame({
    "famille": ["numériques", "catégorielles", "exclues", "cible"],
    "nombre": [len(numeric_features), len(categorical_features), len(excluded_columns), 1],
    "colonnes": [numeric_features, categorical_features, excluded_columns, [target_column]],
})

display(variables_resume)
print("Distribution de la cible :")
display(y.value_counts(normalize=True).sort_index().rename("proportion"))

## 11. Analyse exploratoire après nettoyage

Cette analyse rapide vérifie les relations simples entre les variables disponibles pour le modèle et la satisfaction client.

Les variables post-séjour ne sont pas utilisées dans cette analyse de modélisation.

In [ ]:
correlation_columns = [column for column in numeric_features if column != target_column]
correlation_df = df_model[correlation_columns + [target_column]].copy()

spearman_target = (
    correlation_df.corr(method="spearman")[target_column]
    .drop(target_column)
    .sort_values(key=lambda serie: serie.abs(), ascending=False)
)

display(spearman_target.to_frame("correlation_spearman_satisfaction"))

In [ ]:
for column in ["client_type", "destination", "saison", "type_hebergement", "meteo_prevue", "activite_principale"]:
    if column in df_model.columns:
        print(f"\nSatisfaction moyenne par {column}")
        display(
            df_model.groupby(column, dropna=False)[target_column]
            .agg(nb_sejours="count", satisfaction_moyenne="mean")
            .round(2)
            .sort_values("satisfaction_moyenne", ascending=False)
        )

## 12. Pipeline de transformation

Le pipeline applique les traitements suivants :

- variables numériques : imputation par médiane puis standardisation ;
- variables catégorielles : imputation par `inconnu` puis encodage One-Hot.

Ces transformations sont intégrées dans un `ColumnTransformer` afin d'être apprises uniquement sur le jeu d'entraînement.
Même si les valeurs manquantes ont déjà été traitées dans `df_model`, les imputers sont conservés dans le pipeline par robustesse, notamment pour traiter de futures données en production.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="inconnu"),
    OneHotEncoder(handle_unknown="ignore"),
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

preprocessor

## 13. Modèle baseline

Le premier modèle est une régression logistique multi-classes.

Ce modèle sert de baseline : il donne un premier niveau de performance et permet de vérifier que la chaîne de préparation fonctionne correctement.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

baseline_pipeline = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)

baseline_pipeline.fit(X_train, y_train)
y_pred = baseline_pipeline.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy baseline : {baseline_accuracy:.4f}")

In [ ]:
print(classification_report(y_test, y_pred, labels=class_labels))

In [ ]:
confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=class_labels),
    index=[f"réel_{label}" for label in class_labels],
    columns=[f"prédit_{label}" for label in class_labels],
)

display(confusion)

plt.figure(figsize=(6, 5))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues")
plt.title("Matrice de confusion — baseline")
plt.tight_layout()
plt.show()

## 14. Modélisation et comparaison de modèles

La baseline permet de vérifier que la chaîne de traitement fonctionne. L'étape de modélisation compare maintenant plusieurs modèles afin d'identifier celui qui donne le meilleur compromis sur le dataset nettoyé.

Les modèles testés restent volontairement légers pour limiter la consommation mémoire :

- `DummyClassifier` : modèle naïf de référence ;
- `LogisticRegression` : modèle linéaire interprétable ;
- `RandomForestClassifier` : modèle non linéaire robuste ;
- `ExtraTreesClassifier` : variante d'arbres plus aléatoire et souvent rapide.

Les métriques utilisées sont :

- `accuracy` : part totale de prédictions correctes ;
- `balanced_accuracy` : accuracy corrigée pour le déséquilibre des classes ;
- `macro_f1` : moyenne des F1-scores par classe, utile quand toutes les notes de satisfaction doivent être considérées.

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score

model_candidates = {
    "Dummy_majority": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=80,
        max_depth=6,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=80,
        max_depth=6,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    ),
}

model_results = []
trained_models = {}
model_predictions = {}

for model_name, estimator in model_candidates.items():
    model_pipeline = make_pipeline(
        clone(preprocessor),
        estimator,
    )
    model_pipeline.fit(X_train, y_train)
    predictions_model = model_pipeline.predict(X_test)

    trained_models[model_name] = model_pipeline
    model_predictions[model_name] = predictions_model

    model_results.append({
        "modele": model_name,
        "accuracy": accuracy_score(y_test, predictions_model),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions_model),
        "macro_f1": f1_score(y_test, predictions_model, average="macro"),
    })

model_comparison = (
    pd.DataFrame(model_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

model_comparison[["accuracy", "balanced_accuracy", "macro_f1"]] = model_comparison[[
    "accuracy", "balanced_accuracy", "macro_f1"
]].round(4)

display(model_comparison)

In [ ]:
comparison_plot = model_comparison.set_index("modele")[["accuracy", "balanced_accuracy", "macro_f1"]]

ax = comparison_plot.plot(kind="bar", figsize=(9, 5))
ax.set_title("Comparaison des modèles")
ax.set_ylabel("Score")
ax.set_ylim(0, max(0.35, comparison_plot.max().max() + 0.05))
ax.tick_params(axis="x", rotation=30)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Lecture des métriques

Le modèle `Dummy_majority` peut obtenir une accuracy relativement élevée s'il prédit toujours la classe la plus fréquente. Cette performance est trompeuse, car il ignore les autres classes de satisfaction.

C'est pourquoi la sélection du meilleur modèle se fait avec `macro_f1`, qui évalue mieux la capacité du modèle à traiter toutes les classes.


### Impact de l'enrichissement métier

Pour vérifier l'intérêt de l'enrichissement, on compare les mêmes modèles avec deux jeux de variables :

- **sans enrichissement** : variables initiales et features métier créées ;
- **avec enrichissement** : mêmes variables + contexte destination ajouté manuellement.

La comparaison se fait sur le même découpage train/test pour éviter un biais lié à l'échantillonnage.


In [ ]:
def build_preprocessor_for(features_dataframe):
    numeric_columns = features_dataframe.select_dtypes(include="number").columns.tolist()
    categorical_columns = features_dataframe.select_dtypes(exclude="number").columns.tolist()

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ]
    )

base_feature_columns = [column for column in feature_columns if column not in colonnes_enrichies]
X_base = X[base_feature_columns].copy()
X_enriched = X[feature_columns].copy()

X_train_base = X_base.loc[X_train.index]
X_test_base = X_base.loc[X_test.index]
X_train_enriched = X_enriched.loc[X_train.index]
X_test_enriched = X_enriched.loc[X_test.index]

impact_model_candidates = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=80,
        max_depth=6,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=80,
        max_depth=6,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    ),
}

impact_rows = []

for dataset_name, X_train_current, X_test_current in [
    ("sans_enrichissement", X_train_base, X_test_base),
    ("avec_enrichissement", X_train_enriched, X_test_enriched),
]:
    for model_name, estimator in impact_model_candidates.items():
        current_pipeline = make_pipeline(
            build_preprocessor_for(X_train_current),
            clone(estimator),
        )
        current_pipeline.fit(X_train_current, y_train)
        current_predictions = current_pipeline.predict(X_test_current)

        impact_rows.append({
            "jeu_variables": dataset_name,
            "modele": model_name,
            "accuracy": accuracy_score(y_test, current_predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, current_predictions),
            "macro_f1": f1_score(y_test, current_predictions, average="macro"),
        })

enrichment_impact_comparison = pd.DataFrame(impact_rows)
enrichment_impact_comparison[["accuracy", "balanced_accuracy", "macro_f1"]] = enrichment_impact_comparison[[
    "accuracy", "balanced_accuracy", "macro_f1"
]].round(4)

display(enrichment_impact_comparison.sort_values(["modele", "jeu_variables"]))

impact_pivot = enrichment_impact_comparison.pivot(index="modele", columns="jeu_variables", values="macro_f1")
impact_pivot["gain_macro_f1"] = impact_pivot["avec_enrichissement"] - impact_pivot["sans_enrichissement"]
impact_pivot = impact_pivot.round(4).sort_values("gain_macro_f1", ascending=False)

display(impact_pivot)

### Interprétation de l'impact

Après traitement des valeurs manquantes et des valeurs aberrantes, l'enrichissement métier apporte un gain limité pour `RandomForest` :

- sans enrichissement : `macro_f1 = 0.1922` ;
- avec enrichissement : `macro_f1 = 0.2150` ;
- gain observé : `+0.0228`.

Ce gain montre que l'ajout de contexte métier peut aider certains modèles non linéaires, mais l'amélioration reste faible.

Le traitement des valeurs aberrantes rend les données plus cohérentes, mais il ne crée pas de nouveau signal métier fort. L'enrichissement manuel par destination ne suffit donc pas à résoudre le problème de performance.

Cela peut s'expliquer par le fait que la variable `destination` était déjà présente dans le dataset. Les nouvelles variables ajoutent du contexte, mais une partie de cette information était déjà indirectement captée par l'encodage One-Hot de la destination.

Un enrichissement réellement plus utile devra ajouter des informations plus fines et plus proches de l'expérience client : qualité de l'hôtel, durée du vol, nombre d'escales, météo réelle, avis clients, historique client, qualité des activités et niveau de service.

### Sélection du meilleur modèle

Le modèle retenu est sélectionné selon `macro_f1`, car cette métrique évalue la performance moyenne sur toutes les classes de satisfaction.

Cette approche évite de favoriser uniquement la classe majoritaire.

In [ ]:
best_model_name = model_comparison.iloc[0]["modele"]
best_model = trained_models[best_model_name]
best_predictions = model_predictions[best_model_name]

print(f"Meilleur modèle selon macro_f1 : {best_model_name}")
print("\nRapport de classification du meilleur modèle :")
print(classification_report(y_test, best_predictions, labels=class_labels))

In [ ]:
best_confusion = pd.DataFrame(
    confusion_matrix(y_test, best_predictions, labels=class_labels),
    index=[f"réel_{label}" for label in class_labels],
    columns=[f"prédit_{label}" for label in class_labels],
)

display(best_confusion)

plt.figure(figsize=(6, 5))
sns.heatmap(best_confusion, annot=True, fmt="d", cmap="Greens")
plt.title(f"Matrice de confusion — {best_model_name}")
plt.tight_layout()
plt.show()

### Interprétation détaillée des résultats de modélisation

Les résultats obtenus montrent que le pipeline de modélisation fonctionne correctement sur le plan technique :

- les données sont chargées et nettoyées ;
- les incohérences critiques sont traitées ;
- les valeurs manquantes restantes sont imputées ;
- les valeurs aberrantes numériques sont traitées avec la méthode IQR ;
- les variables post-séjour sont exclues pour éviter la fuite de données ;
- les variables numériques et catégorielles sont transformées dans un pipeline ;
- plusieurs modèles sont entraînés et comparés sur un jeu de test.

Après préparation complète et enrichissement métier, le meilleur modèle multi-classes est `RandomForest`.

#### Lecture des résultats

`RandomForest` obtient environ :

- `accuracy = 0.2218` ;
- `balanced_accuracy = 0.2316` ;
- `macro_f1 = 0.2150`.

Le score reste faible malgré le nettoyage et l'enrichissement.

Comme la cible `satisfaction_client` contient 5 classes possibles (`1`, `2`, `3`, `4`, `5`), une performance proche de 22 % indique que le modèle distingue encore difficilement les niveaux de satisfaction.

#### Pourquoi le modèle naïf peut sembler compétitif

Le modèle `Dummy_majority` prédit toujours la classe la plus fréquente.

Il obtient une accuracy d'environ `0.2887`, mais cette performance est trompeuse :

- il ne comprend aucune relation entre les variables et la satisfaction ;
- il ignore les classes minoritaires ;
- son `macro_f1` est très faible ;
- il ne permet pas de personnaliser les recommandations.

C'est pourquoi la sélection du meilleur modèle se fait avec `macro_f1`, et non uniquement avec l'accuracy.

#### Interprétation métier

Le traitement des valeurs manquantes et des valeurs aberrantes améliore la qualité du dataset, mais il ne suffit pas à créer un signal prédictif fort.

L'enrichissement par destination améliore légèrement la performance de `RandomForest`, mais il ne suffit pas à obtenir un modèle robuste.

Les variables disponibles avant le séjour ne suffisent donc pas encore à expliquer précisément la satisfaction client.

La satisfaction dépend probablement de facteurs absents du dataset actuel, par exemple :

- qualité réelle de l'hébergement ;
- niveau de service fourni par l'agence ;
- durée du vol et nombre d'escales ;
- ponctualité réelle du transport ;
- météo réellement observée ;
- adéquation entre les préférences client et les activités proposées ;
- historique client ;
- rapport qualité-prix perçu ;
- qualité des prestataires locaux.

#### Conclusion sur la performance

Le modèle est **fonctionnel techniquement**, mais il n'est **pas performant métier** à ce stade.

Les traitements de qualité de données sont nécessaires, mais ils ne remplacent pas un enrichissement métier plus profond.

Le modèle doit donc être considéré comme une baseline enrichie exploratoire, pas comme une solution prête pour la production.

## 15. Test de classification binaire

La prédiction exacte de `satisfaction_client` sur 5 classes est difficile et donne des scores faibles.

Pour vérifier si le problème vient de la granularité de la cible, on teste une classification binaire plus simple :

- `0` : satisfaction faible ou moyenne (`satisfaction_client` de 1 à 3) ;
- `1` : satisfaction élevée (`satisfaction_client` de 4 à 5).

Cette formulation est plus proche d'un besoin métier opérationnel : identifier les séjours susceptibles de générer une satisfaction élevée.

In [ ]:
df_binary = df_model.copy()
df_binary["satisfaction_elevee"] = (df_binary["satisfaction_client"] >= 4).astype(int)

X_binary = df_binary[feature_columns].copy()
y_binary = df_binary["satisfaction_elevee"].copy()

binary_target_distribution = y_binary.value_counts(normalize=True).sort_index().rename("proportion")

display(pd.DataFrame({
    "classe": ["0 = satisfaction 1 à 3", "1 = satisfaction 4 à 5"],
    "effectif": y_binary.value_counts().sort_index().values,
    "proportion": binary_target_distribution.values.round(4),
}))

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score

X_train_binary, X_test_binary, y_train_binary, y_test_binary = train_test_split(
    X_binary,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary,
)

binary_model_candidates = {
    "Dummy_majority": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=80,
        max_depth=6,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    ),
}

binary_results = []
binary_trained_models = {}
binary_predictions = {}

for model_name, estimator in binary_model_candidates.items():
    binary_pipeline = make_pipeline(
        clone(preprocessor),
        estimator,
    )
    binary_pipeline.fit(X_train_binary, y_train_binary)
    predictions_binary = binary_pipeline.predict(X_test_binary)

    if hasattr(binary_pipeline, "predict_proba"):
        positive_scores = binary_pipeline.predict_proba(X_test_binary)[:, 1]
        roc_auc = roc_auc_score(y_test_binary, positive_scores)
    else:
        roc_auc = np.nan

    binary_trained_models[model_name] = binary_pipeline
    binary_predictions[model_name] = predictions_binary

    binary_results.append({
        "modele": model_name,
        "accuracy": accuracy_score(y_test_binary, predictions_binary),
        "balanced_accuracy": balanced_accuracy_score(y_test_binary, predictions_binary),
        "precision_classe_1": precision_score(y_test_binary, predictions_binary, zero_division=0),
        "recall_classe_1": recall_score(y_test_binary, predictions_binary, zero_division=0),
        "f1_classe_1": f1_score(y_test_binary, predictions_binary, zero_division=0),
        "roc_auc": roc_auc,
    })

binary_model_comparison = (
    pd.DataFrame(binary_results)
    .sort_values("f1_classe_1", ascending=False)
    .reset_index(drop=True)
)

score_columns = ["accuracy", "balanced_accuracy", "precision_classe_1", "recall_classe_1", "f1_classe_1", "roc_auc"]
binary_model_comparison[score_columns] = binary_model_comparison[score_columns].round(4)

display(binary_model_comparison)

In [ ]:
best_binary_model_name = binary_model_comparison.iloc[0]["modele"]
best_binary_model = binary_trained_models[best_binary_model_name]
best_binary_predictions = binary_predictions[best_binary_model_name]

print(f"Meilleur modèle binaire selon f1_classe_1 : {best_binary_model_name}")
print("\nRapport de classification binaire :")
print(classification_report(y_test_binary, best_binary_predictions, target_names=["satisfaction_1_3", "satisfaction_4_5"]))

binary_confusion = pd.DataFrame(
    confusion_matrix(y_test_binary, best_binary_predictions, labels=[0, 1]),
    index=["réel_1_3", "réel_4_5"],
    columns=["prédit_1_3", "prédit_4_5"],
)

display(binary_confusion)

plt.figure(figsize=(5, 4))
sns.heatmap(binary_confusion, annot=True, fmt="d", cmap="Purples")
plt.title(f"Matrice de confusion binaire — {best_binary_model_name}")
plt.tight_layout()
plt.show()

### Interprétation de la classification binaire

La classification binaire simplifie le problème : au lieu de prédire précisément une note de `1` à `5`, le modèle cherche à distinguer :

- `0` : satisfaction faible ou moyenne (`1`, `2`, `3`) ;
- `1` : satisfaction élevée (`4`, `5`).

Après traitement des valeurs manquantes, traitement des valeurs aberrantes et enrichissement, `RandomForest` devient le meilleur modèle binaire selon le `f1_classe_1`.

#### Lecture des résultats obtenus

Le modèle naïf `Dummy_majority` obtient une accuracy élevée car il prédit toujours la classe majoritaire `0`.

Cette accuracy est trompeuse :

- son `recall_classe_1` vaut `0` ;
- son `f1_classe_1` vaut `0` ;
- il ne détecte aucun séjour à satisfaction élevée.

`RandomForest` obtient environ :

- `accuracy = 0.5845` ;
- `balanced_accuracy = 0.5579` ;
- `f1_classe_1 = 0.4100` ;
- `roc_auc = 0.5355`.

Ces scores sont meilleurs que ceux de la classification en 5 classes, mais ils restent insuffisants pour une décision métier fiable.

#### Conclusion métier

Le passage en classification binaire améliore la lisibilité du problème et permet de mieux détecter les séjours à satisfaction élevée.

Cependant, les scores restent modestes. Cela confirme que le dataset actuel ne contient pas assez de variables explicatives fortes pour prédire correctement la satisfaction client, même avec une cible simplifiée, un nettoyage complet et un enrichissement métier simple.

L'enrichissement des données reste donc nécessaire avant d'espérer un modèle réellement performant.

## 16. Expérience à part : régression linéaire

La variable `satisfaction_client` est une note ordonnée de `1` à `5`. On peut donc tester une approche par régression linéaire en considérant la satisfaction comme une valeur numérique continue.

Cette expérience est conservée à part, car le problème principal reste formulé comme une classification. La régression linéaire permet surtout de vérifier si les variables disponibles expliquent une variation continue de la satisfaction.

Les résultats seront évalués avec :

- `MAE` : erreur absolue moyenne en points de satisfaction ;
- `RMSE` : erreur quadratique moyenne, plus sensible aux grosses erreurs ;
- `R²` : part de variance expliquée par le modèle ;
- prédictions arrondies entre `1` et `5` pour comparer avec les métriques de classification.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

linear_regression_pipeline = make_pipeline(
    clone(preprocessor),
    LinearRegression(),
)

linear_regression_pipeline.fit(X_train, y_train)
y_pred_regression = linear_regression_pipeline.predict(X_test)

regression_mae = mean_absolute_error(y_test, y_pred_regression)
regression_rmse = np.sqrt(mean_squared_error(y_test, y_pred_regression))
regression_r2 = r2_score(y_test, y_pred_regression)

regression_metrics = pd.DataFrame([
    {
        "modele": "LinearRegression",
        "MAE": regression_mae,
        "RMSE": regression_rmse,
        "R2": regression_r2,
    }
])

regression_metrics[["MAE", "RMSE", "R2"]] = regression_metrics[["MAE", "RMSE", "R2"]].round(4)
display(regression_metrics)

In [ ]:
y_pred_regression_rounded = (
    np.rint(y_pred_regression)
    .clip(1, 5)
    .astype(int)
)

regression_as_classification = pd.DataFrame([
    {
        "modele": "LinearRegression arrondie",
        "accuracy": accuracy_score(y_test, y_pred_regression_rounded),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred_regression_rounded),
        "macro_f1": f1_score(y_test, y_pred_regression_rounded, average="macro"),
    }
])

regression_as_classification[["accuracy", "balanced_accuracy", "macro_f1"]] = regression_as_classification[[
    "accuracy", "balanced_accuracy", "macro_f1"
]].round(4)

display(regression_as_classification)

regression_confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred_regression_rounded, labels=class_labels),
    index=[f"réel_{label}" for label in class_labels],
    columns=[f"prédit_{label}" for label in class_labels],
)

display(regression_confusion)

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_test, y=y_pred_regression, alpha=0.6)
plt.plot([1, 5], [1, 5], color="red", linestyle="--", label="prédiction parfaite")
plt.xlabel("Satisfaction réelle")
plt.ylabel("Satisfaction prédite")
plt.title("Régression linéaire — valeurs réelles vs prédites")
plt.legend()
plt.tight_layout()
plt.show()

### Interprétation de la régression linéaire

La régression linéaire obtient les résultats suivants :

- `MAE = 1.0464` ;
- `RMSE = 1.2607` ;
- `R² = 0.0007`.

#### Lecture métier

Le `MAE` indique que le modèle se trompe en moyenne d'environ **1 point de satisfaction** sur une échelle allant de `1` à `5`.

Le `R²` est quasiment nul. Cela signifie que le modèle n'explique pratiquement pas la variation de la satisfaction client. En pratique, il ne fait pas mieux qu'une prédiction très simple basée sur la moyenne.

#### Comparaison après arrondi

Lorsque les prédictions continues sont arrondies entre `1` et `5`, la régression linéaire obtient :

- `accuracy = 0.2746` ;
- `balanced_accuracy = 0.2239` ;
- `macro_f1 = 0.1337`.

L'accuracy semble proche du modèle naïf, mais le `macro_f1` reste faible. Cela montre que le modèle prédit mal les différentes classes de satisfaction.

#### Conclusion

Cette expérience confirme que la satisfaction client n'est pas bien expliquée par une relation linéaire avec les variables disponibles.

La régression linéaire ne constitue donc pas un modèle performant pour ce dataset. Elle renforce la conclusion précédente : sans enrichissement des données plus fin, le modèle restera limité, que l'on formule le problème en classification multi-classes, en classification binaire ou en régression.

## 17. Visualisations de contrôle

Ces graphiques permettent de vérifier rapidement la distribution des principales variables après nettoyage.

In [ ]:
plot_columns = [
    column
    for column in ["budget_total", "prix_vol", "duree_jours", "budget_par_jour", "part_vol_budget", "satisfaction_client"]
    if column in df_model.columns
]

fig, axes = plt.subplots(len(plot_columns), 1, figsize=(8, 3 * len(plot_columns)))
if len(plot_columns) == 1:
    axes = [axes]

for axis, column in zip(axes, plot_columns):
    sns.histplot(df_model[column], kde=True, ax=axis)
    axis.set_title(column)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
correlation_matrix = df_model[numeric_features + [target_column]].corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="viridis")
plt.title("Corrélations numériques après nettoyage")
plt.tight_layout()
plt.show()

## 18. Conclusion du notebook

Le notebook met en place une chaîne complète et cohérente pour un premier prototype IA :

- chargement du dataset brut ;
- identification des incohérences critiques ;
- suppression des lignes sans cible exploitable ou avec incohérence budgétaire forte ;
- traitement des valeurs manquantes restantes ;
- traitement des valeurs aberrantes numériques ;
- exclusion des variables post-séjour pour éviter la fuite de données ;
- création de variables métier ;
- enrichissement métier local par destination ;
- préparation des variables numériques et catégorielles dans un pipeline ;
- entraînement et évaluation d'un modèle baseline ;
- comparaison de plusieurs modèles légers et sélection selon `macro_f1` ;
- test d'une classification binaire pour vérifier si une cible simplifiée améliore les résultats ;
- expérience de régression linéaire pour vérifier si la satisfaction peut être expliquée comme une variable numérique continue.

Les résultats montrent que le nettoyage améliore la cohérence du dataset et que l'enrichissement manuel améliore légèrement certains modèles, mais pas assez pour atteindre une performance métier fiable. Le modèle doit donc être considéré comme une baseline enrichie exploratoire. Les prochaines étapes devront prioriser un enrichissement plus fin des données et une validation métier de la cible.